## Објаснување за функцијата и видеата
се користи easyocr на српска кирилица (како fallback e ставен бугарски и потоа англиски јазик)

Captions НЕ се 100% точни, но колку е поточено скратено видеото за да се само caprions толку се истите подобри. Во однос на тоа, следните вредности:

    top
    bottom
    left
    right
  
се користат на следниот начин:

примаат децимални вредности од 0 - 1, со тоа што ако кажеме top = 0.5, видеото се крати од горе 50%, со што останува само долната половина од видеото.

### Сите пробани видеа се youtube shorts.

In [58]:
!pip install -q yt-dlp easyocr opencv-python-headless rapidfuzz

import cv2
import easyocr
import yt_dlp
from rapidfuzz.fuzz import ratio

reader = easyocr.Reader(['rs_cyrillic', 'bg', 'en'])

def extract_captions(
    youtube_url,
    output_txt="captions.txt",
    sharpen=True,
    frame_every_seconds=0.5,
    similarity_threshold=85,
    top=0.65,
    bottom=1.0,
    left=0.0,
    right=1.0,
    save_debug=False
):

    video_path = "temp_video.mp4"

    ydl_opts = {
        'format': 'mp4',
        'outtmpl': video_path,
        'quiet': True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

    print("Video downloaded!")

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_interval = max(
        1,
        int(fps * frame_every_seconds)
    )

    frame_count = 0

    results = []

    last_text = ""

    debug_saved = False

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % frame_interval == 0:

            frame = cv2.resize(
                frame,
                None,
                fx=2,
                fy=2,
                interpolation=cv2.INTER_CUBIC
            )

            h, w, _ = frame.shape

            y1 = int(h * top)
            y2 = int(h * bottom)

            x1 = int(w * left)
            x2 = int(w * right)

            crop = frame[y1:y2, x1:x2]

            if save_debug and not debug_saved:

                debug_frame = frame.copy()

                cv2.rectangle(
                    debug_frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    4
                )

                cv2.imwrite(
                    "debug_region.jpg",
                    debug_frame
                )

                cv2.imwrite(
                    "debug_crop.jpg",
                    crop
                )

                debug_saved = True

            image_for_ocr = crop

            if sharpen:

                gray = cv2.cvtColor(
                    crop,
                    cv2.COLOR_BGR2GRAY
                )

                gray = cv2.threshold(
                    gray,
                    180,
                    255,
                    cv2.THRESH_BINARY
                )[1]

                image_for_ocr = gray

            text_results = reader.readtext(
                image_for_ocr
            )

            texts = []

            for r in text_results:

                text = r[1]
                confidence = r[2]

                if confidence < 0.25:
                    continue

                texts.append(text)

            combined = " ".join(texts).strip()

            combined = combined.replace("\n", " ")
            combined = " ".join(combined.split())

            if len(combined) > 2:

                similarity = ratio(
                    combined,
                    last_text
                )

                if similarity < similarity_threshold:

                    timestamp = round(
                        frame_count / fps,
                        2
                    )

                    results.append({
                        "time": timestamp,
                        "text": combined
                    })

                    print(
                        f"[{timestamp}s] {combined}"
                    )

                    last_text = combined

        frame_count += 1

    cap.release()

    with open(
        output_txt,
        "w",
        encoding="utf-8"
    ) as f:

        for r in results:

            f.write(
                f"[{r['time']}s] {r['text']}\n"
            )

    print(f"\nSaved to {output_txt}")

    return results

## Видеа од подкаст

In [54]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/OFmFfJcet3Q",
    top=0.7,
    bottom=0.95,
    left=0.1,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[2.5s] СЕЗОНСКИТЕ АЛЕРГИИ МОРА
[4.5s] ДАСЕ ТРЕТИРААТ СО
[5.0s] ДА CE ТРЕТИРААТСО
[5.5s] ИСКЛУЧИТЕЛНА ВНИМАТЕЛНОСТ,.
[7.5s] БИДЕЈКИ,
[8.5s] ЧЕСТО СИМПТОМИТЕ
[10.0s] ПОМЕГУ АЛЕРГИЈА,
[11.0s] ВИРУС И НАСТИНКА
[13.0s] ЗНААТДА БИДАТ
[14.0s] СЛИЧНИ; АKO НЕЕИ
[15.0s] ИДЕНТИЧНИ: ОСВЕН
[16.5s] СЕЗОНСКИ АЛЕРГИИ,
[18.0s] ПРИСУТНV CЕ ИАЛЕРГИИ НА
[20.0s] КОЖА,
[21.0s] АЛЕРГИVОД ДОМАШНА
[23.0s] ПРАШИНА, ДОМАШНИ
[24.0s] МИЛЕНИЧИЊА KAKO И
[26.0s] АЛЕРГИV НА ХРАНАИ
[27.5s] ОДРЕДЕНИ ЛЕКОВИ:
[28.5s] MOPAN ДА НАПОМЕНАМ
[29.0s] МОРАМДА НАПОМЕНАМ
[30.0s] ДЕКА НЕКОГАШ АЛЕРГИСКА
[31.5s] РЕАКЦИЈА МОЖЕ ДА ДАДЕ
[33.0s] ИОДРЕДЕНА ХРАНА KOJA И
[33.5s] И ОДРЕДЕНА ХРАНА КОЈA И
[34.5s] СЕКОЈДНЕВНО CMEJA
[35.5s] КОРИСТЕЛЕ ИЛИ
[37.0s] ЛЕКОВИ кОИ
[37.5s] ЛЕКОВVкоИ
[38.0s] ЛЕКОВИКОИ
[39.0s] НЕ ГИЗЕМАМЕ ЗА ПРВ ПАТ
[39.5s] HE ГИ ЗЕМАМЕ ЗАПРЕ ПАТ
[40.0s] ВОНАШИОТ ЖИВОТ ПОКРАЈ
[41.5s] ОНВЕНЦИОНАЛНИ Е МЕТОДИ
[43.5s] КОИШтосЕ КОРИСТАТ,
[44.5s] ПОКРАЈ: ЛЕКОВИТЕ,
[45.5s] Односно
[46.0s] АНТИХИСТАМИНИЦИ TE
[47.5s] ПОСТОЈАТ,
[48.5

[{'time': 2.5, 'text': 'СЕЗОНСКИТЕ АЛЕРГИИ МОРА'},
 {'time': 4.5, 'text': 'ДАСЕ ТРЕТИРААТ СО'},
 {'time': 5.0, 'text': 'ДА CE ТРЕТИРААТСО'},
 {'time': 5.5, 'text': 'ИСКЛУЧИТЕЛНА ВНИМАТЕЛНОСТ,.'},
 {'time': 7.5, 'text': 'БИДЕЈКИ,'},
 {'time': 8.5, 'text': 'ЧЕСТО СИМПТОМИТЕ'},
 {'time': 10.0, 'text': 'ПОМЕГУ АЛЕРГИЈА,'},
 {'time': 11.0, 'text': 'ВИРУС И НАСТИНКА'},
 {'time': 13.0, 'text': 'ЗНААТДА БИДАТ'},
 {'time': 14.0, 'text': 'СЛИЧНИ; АKO НЕЕИ'},
 {'time': 15.0, 'text': 'ИДЕНТИЧНИ: ОСВЕН'},
 {'time': 16.5, 'text': 'СЕЗОНСКИ АЛЕРГИИ,'},
 {'time': 18.0, 'text': 'ПРИСУТНV CЕ ИАЛЕРГИИ НА'},
 {'time': 20.0, 'text': 'КОЖА,'},
 {'time': 21.0, 'text': 'АЛЕРГИVОД ДОМАШНА'},
 {'time': 23.0, 'text': 'ПРАШИНА, ДОМАШНИ'},
 {'time': 24.0, 'text': 'МИЛЕНИЧИЊА KAKO И'},
 {'time': 26.0, 'text': 'АЛЕРГИV НА ХРАНАИ'},
 {'time': 27.5, 'text': 'ОДРЕДЕНИ ЛЕКОВИ:'},
 {'time': 28.5, 'text': 'MOPAN ДА НАПОМЕНАМ'},
 {'time': 29.0, 'text': 'МОРАМДА НАПОМЕНАМ'},
 {'time': 30.0, 'text': 'ДЕКА НЕКОГАШ АЛЕРГИСКА'}

In [56]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/CyZMKGzNidk",
    top=0.6,
    bottom=0.85,
    left=0.1,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.5s] ЖИВЕЕМЕВО ОПШТЕСТВО CO
[1.5s] ПРЕМНОГУ НАСИЛСТВО ВРЗ
[4.0s] ЖЕНИ: ПРЕМНОГУ
[6.0s] СЕКСУАЛНО НАСИЛСТВО ,
[7.0s] ПРЕМНОГУ ОПРЕСИРАЧКИ
[11.0s] КУЛТУРНИ ОДНЕСУВАЊА:
[12.0s] ГО КАЖАМ НА
[12.5s] ЕВЕ ТАКА КE ГОКАЖАNНА
[13.0s] ПРЕМНОГУ
[13.5s] ОПШТО НИВО. ПРЕМНОГУ
[15.5s] НАМЕТНАТИ:. JAC HE JA
[18.5s] РАЗБИРАМ ПОТРЕБАТАТИ
[19.5s] НАНЕКОГСДА МУ НАМЕТНУВАШ
[20.5s] ДА СТАНЕ РОДИТЕЛ:
[22.5s] ПОСЕБНС НА ЖЕНИ:
[24.5s] ЗОШТО ПОСЕБНС НА
[25.0s] ЖЕНИ? ЖЕНСКОТО СИ
[26.0s] ГО РИЗИКУВА ЖИВОТОТ ЗА ДА
[27.5s] ДОНЕСЕ НЕКОГО НА
[28.5s] CBЕТ: ЈАC СУМ
[32.0s] ТОЛКУ ПОВРЕДЕНА,
[32.5s] ТОЛКУ ИСПЛАШЕНА
[33.5s] КОГА ГЛЕДАМ СИТУАЦИИ
[35.5s] KAJ ШТО РОДИЛКИ НА
[36.5s] МОЈАВОЗРАСТ УМИРААТ НА
[37.0s] MOJA ВОЗРАСТ УМИРААТНА
[39.0s] ПОРОДИЛНО:
[42.5s] |м otcag

Saved to captions.txt


[{'time': 0.5, 'text': 'ЖИВЕЕМЕВО ОПШТЕСТВО CO'},
 {'time': 1.5, 'text': 'ПРЕМНОГУ НАСИЛСТВО ВРЗ'},
 {'time': 4.0, 'text': 'ЖЕНИ: ПРЕМНОГУ'},
 {'time': 6.0, 'text': 'СЕКСУАЛНО НАСИЛСТВО ,'},
 {'time': 7.0, 'text': 'ПРЕМНОГУ ОПРЕСИРАЧКИ'},
 {'time': 11.0, 'text': 'КУЛТУРНИ ОДНЕСУВАЊА:'},
 {'time': 12.0, 'text': 'ГО КАЖАМ НА'},
 {'time': 12.5, 'text': 'ЕВЕ ТАКА КE ГОКАЖАNНА'},
 {'time': 13.0, 'text': 'ПРЕМНОГУ'},
 {'time': 13.5, 'text': 'ОПШТО НИВО. ПРЕМНОГУ'},
 {'time': 15.5, 'text': 'НАМЕТНАТИ:. JAC HE JA'},
 {'time': 18.5, 'text': 'РАЗБИРАМ ПОТРЕБАТАТИ'},
 {'time': 19.5, 'text': 'НАНЕКОГСДА МУ НАМЕТНУВАШ'},
 {'time': 20.5, 'text': 'ДА СТАНЕ РОДИТЕЛ:'},
 {'time': 22.5, 'text': 'ПОСЕБНС НА ЖЕНИ:'},
 {'time': 24.5, 'text': 'ЗОШТО ПОСЕБНС НА'},
 {'time': 25.0, 'text': 'ЖЕНИ? ЖЕНСКОТО СИ'},
 {'time': 26.0, 'text': 'ГО РИЗИКУВА ЖИВОТОТ ЗА ДА'},
 {'time': 27.5, 'text': 'ДОНЕСЕ НЕКОГО НА'},
 {'time': 28.5, 'text': 'CBЕТ: ЈАC СУМ'},
 {'time': 32.0, 'text': 'ТОЛКУ ПОВРЕДЕНА,'},
 {'time': 32.5, 

## Видеа од Емисија “На кавга со Иван“ (Канал5 телевизија)

Првиот пример е „поточен“, бидејќи е подобро скратен просторот за да се само captions. Следните 2 ќелии се од истото видео, само за да се спореди резултатот со различни атрибути за top, bottom вредностите.  

In [ ]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/nNyVy1eFzCs",
    top=0.65,
    bottom=0.75,
    left=0.05,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.0s] СОЦИЈАЛДЕМОКРАТСКИТЕ ВРЕДНОСТИ
[1.4s] СМЕТАМ ДЕКА ВO МОМЕНТОГ СЕ"ПОМИШТЕНИ
[1.87s] СМЕТАМДЕКА ВO MOМЕНТОL CE ПОТИШТЕНИ
[4.2s] ТЕШКО Е ДА CЕ РАЗБЕРАТ BO OВOJ' МОМЕНТ,
[5.14s] ТЕШКО Е ДАСЕРАЗБЕРАТ BO ОВОЈМОМЕНТ,
[8.41s] НИКОГО НЕППОТЦЕНУВАМ ДЕКА HЕ МОЖЕ'
[10.74s] ЗАТОА ШiТО МОЖЕБИ; И НИЕ' HE CME
[12.15s] ГИ ДООБЈАСНИЛЕ КАКО ШТО ТРЕБА

Saved to captions.txt


[{'time': 0.0, 'text': 'СОЦИЈАЛДЕМОКРАТСКИТЕ ВРЕДНОСТИ'},
 {'time': 1.4, 'text': 'СМЕТАМ ДЕКА ВO МОМЕНТОГ СЕ"ПОМИШТЕНИ'},
 {'time': 1.87, 'text': 'СМЕТАМДЕКА ВO MOМЕНТОL CE ПОТИШТЕНИ'},
 {'time': 4.2, 'text': "ТЕШКО Е ДА CЕ РАЗБЕРАТ BO OВOJ' МОМЕНТ,"},
 {'time': 5.14, 'text': 'ТЕШКО Е ДАСЕРАЗБЕРАТ BO ОВОЈМОМЕНТ,'},
 {'time': 8.41, 'text': "НИКОГО НЕППОТЦЕНУВАМ ДЕКА HЕ МОЖЕ'"},
 {'time': 10.74, 'text': "ЗАТОА ШiТО МОЖЕБИ; И НИЕ' HE CME"},
 {'time': 12.15, 'text': 'ГИ ДООБЈАСНИЛЕ КАКО ШТО ТРЕБА'}]

In [ ]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/IWINFbz9ddg",
    top=0.65,
    bottom=0.8,
    left=0.05,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.0s] ME МАНИПУЛИРАТЕ
[0.93s] МОЈOT СОПРУГ РЕЧЕ ДЕКА JAC СУМ БИЛА
[4.67s] КАДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?
[6.07s] ЗНАЧИ JAC СУМ БИЛА ВO CBOЈСТВО .
[7.94s] ЗНАЧИ; JAC СУМ БИЛА ВО CВOЈCTВО
[8.41s] ЗНАЧИ ЈAC СУМ БИЛА ВO CBOЈCТВO
[8.88s] ЗНАЧИ JAC СУМ
[9.34s] ЗНАЧИ JAC СУМ БИЛА ВO СВOЈCТВО
[10.28s] А ШТО ЗНАЧИ TOА?
[11.21s] ШТо ЗНАЧИ TOA BO СВОЈСТBO?
[12.61s] HE СУМ CE ПРЕТСТАВИЛА КАКО АТАШЕ HЕ СУМ БИЛА
[15.42s] ТАМУКАКО АТАШЕ, ТУКУ СУМ БИЛА КАКО НЕКОЈ
[20.09s] KOЈ ШТО АМБАСАДАТА ГО ДОНЕСЛА:
[24.29s] ШТОМ АМБАСАДАТА ME ДОНЕСЕ ТАМУ
[25.69s] AMA HЕ ТРГНАВ НАТАМУ .
[27.56s] АС"ВЕКЕ HE ГО ПРЕТСТАВУВА МОјOT УНИВЕРЗИТЕТ
[29.43s] АСВЕКЕ НЕЏi@ ПРЕТСТАВУВА МОЈOT УНИВЕРЗИТЕТ
[30.83s] ШУКУЈА ПРЕПСТАВУВАМ МОЈАТАДРЖАВА:
[31.3s] ТУКУ JA ПРЕТСТАВУВАМ МОЈAТA ДРЖАВА:
[33.63s] КОЈ ВИДАЛЬ ЛЕГИТИМИТЕТ ВИЕ ДА ME ПРЕТСТАВУВАТЕ
[36.9s] МЕНЕ КАКО НАУЧНО АТАШЕ:
[38.77s] ЛЕГИТИМИТЕТ. HE НАУЧЕН АТАШЕ
[40.64s] AMA СОСЛУШАЈТЕ ME ОСПОГО НАЈЧЕВСКА.ВАШИОЪ
[43.91s] AMA BЕ МОЛАМ HE ВИ ДОЗВОЛУВАМ; МАНИПУЛИРА-
[47

[{'time': 0.0, 'text': 'ME МАНИПУЛИРАТЕ'},
 {'time': 0.93, 'text': 'МОЈOT СОПРУГ РЕЧЕ ДЕКА JAC СУМ БИЛА'},
 {'time': 4.67, 'text': 'КАДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?'},
 {'time': 6.07, 'text': 'ЗНАЧИ JAC СУМ БИЛА ВO CBOЈСТВО .'},
 {'time': 7.94, 'text': 'ЗНАЧИ; JAC СУМ БИЛА ВО CВOЈCTВО'},
 {'time': 8.41, 'text': 'ЗНАЧИ ЈAC СУМ БИЛА ВO CBOЈCТВO'},
 {'time': 8.88, 'text': 'ЗНАЧИ JAC СУМ'},
 {'time': 9.34, 'text': 'ЗНАЧИ JAC СУМ БИЛА ВO СВOЈCТВО'},
 {'time': 10.28, 'text': 'А ШТО ЗНАЧИ TOА?'},
 {'time': 11.21, 'text': 'ШТо ЗНАЧИ TOA BO СВОЈСТBO?'},
 {'time': 12.61, 'text': 'HE СУМ CE ПРЕТСТАВИЛА КАКО АТАШЕ HЕ СУМ БИЛА'},
 {'time': 15.42, 'text': 'ТАМУКАКО АТАШЕ, ТУКУ СУМ БИЛА КАКО НЕКОЈ'},
 {'time': 20.09, 'text': 'KOЈ ШТО АМБАСАДАТА ГО ДОНЕСЛА:'},
 {'time': 24.29, 'text': 'ШТОМ АМБАСАДАТА ME ДОНЕСЕ ТАМУ'},
 {'time': 25.69, 'text': 'AMA HЕ ТРГНАВ НАТАМУ .'},
 {'time': 27.56, 'text': 'АС"ВЕКЕ HE ГО ПРЕТСТАВУВА МОјOT УНИВЕРЗИТЕТ'},
 {'time': 29.43, 'text': 'АСВЕКЕ НЕЏi@ ПРЕТСТАВУВА 

In [48]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/IWINFbz9ddg",
    top=0.65,
    bottom=0.75,
    left=0.05,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.0s] ME МАНИПУЛИРАТЕ
[0.93s] МОЈOT СОПРУГ РЕЧЕ ДЕКА JAC СУМ БИЛА
[4.67s] КАДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?
[5.14s] ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5? 'ЖАДЕ?
[5.61s] "[АДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?
[6.07s] ЗНАЧИ JAC СУМ БИЛА ВO CВОЈСТВО .
[7.94s] ЗНАЧИ; JAC СУМ БИЛА ВO CBOЈCTВО
[8.41s] ЗНАЧИ ЈAC СУМ
[8.88s] ЗНАЧИ JAC СУМ БИЛА ВO СВOЈCTВО.
[10.28s] А ШТО ЗНАЧИ TOА?
[11.21s] А ШТО ЗНАЧИ TOA BO СВОЈСТВО?
[12.61s] HE СУМ CE ПРЕТСТАВИЛА КАКО АТАШЕ HЕ СУМ БИЛА
[15.42s] ТАМУКАКО АТАШЕ, ТУКУ СУМ БИЛА КАКО НЕКОЈ
[20.09s] KOЈ ШТО АМБАСАДАТА ГО ДОНЕСЛА:
[24.29s] ШТОМ АМБАСАДАТА ME ДОНЕСЕ ТАМУ
[25.69s] AMA HЕ ТРГНАВ НАТАМУ
[27.56s] АС"ВЕКЕ HE ГО ПРЕТСТАВУВА МОјOT УНИВЕРЗИТЕТ
[30.36s] АС{ВЕКЕ НЕl@ ПРЕТСТАВУВА МОЈOT УНИВЕРЗИТЕТ
[30.83s] ШУКУЈА ПРЕПСТАВУВАМ МОЈАТАДРЖАВА:
[31.3s] ТУКУ JA ПРЕТСТАВУВАМ MOЈATA ДРЖАВА:
[33.63s] КОЈ ВИДАЛЬ ЛЕГИТИМИТЕТ ВИЕ ДА ME ПРЕТСТАВУВАТЕ
[36.9s] МЕНЕ КАКО НАУЧНО АТАШЕ:
[38.77s] ЛЕГИТИМИТЕТ. HE НАУЧЕН АТАШЕ
[40.64s] AMA СОСЛУШАЈТЕ ME, ОСПОЙО НАЈЧЕВС

[{'time': 0.0, 'text': 'ME МАНИПУЛИРАТЕ'},
 {'time': 0.93, 'text': 'МОЈOT СОПРУГ РЕЧЕ ДЕКА JAC СУМ БИЛА'},
 {'time': 4.67, 'text': 'КАДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?'},
 {'time': 5.14, 'text': "ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5? 'ЖАДЕ?"},
 {'time': 5.61, 'text': '"[АДЕ? ЗНАЕТЕ ДЕКА ДАДЕ ИЗЈАВА ЗА КАНАЛ 5?'},
 {'time': 6.07, 'text': 'ЗНАЧИ JAC СУМ БИЛА ВO CВОЈСТВО .'},
 {'time': 7.94, 'text': 'ЗНАЧИ; JAC СУМ БИЛА ВO CBOЈCTВО'},
 {'time': 8.41, 'text': 'ЗНАЧИ ЈAC СУМ'},
 {'time': 8.88, 'text': 'ЗНАЧИ JAC СУМ БИЛА ВO СВOЈCTВО.'},
 {'time': 10.28, 'text': 'А ШТО ЗНАЧИ TOА?'},
 {'time': 11.21, 'text': 'А ШТО ЗНАЧИ TOA BO СВОЈСТВО?'},
 {'time': 12.61, 'text': 'HE СУМ CE ПРЕТСТАВИЛА КАКО АТАШЕ HЕ СУМ БИЛА'},
 {'time': 15.42, 'text': 'ТАМУКАКО АТАШЕ, ТУКУ СУМ БИЛА КАКО НЕКОЈ'},
 {'time': 20.09, 'text': 'KOЈ ШТО АМБАСАДАТА ГО ДОНЕСЛА:'},
 {'time': 24.29, 'text': 'ШТОМ АМБАСАДАТА ME ДОНЕСЕ ТАМУ'},
 {'time': 25.69, 'text': 'AMA HЕ ТРГНАВ НАТАМУ'},
 {'time': 27.56, 'text': 'АС"ВЕКЕ HE ГО 

## Видеа од каналот @trn_mk

Во овие видеа captions не се толку јасно одделени / едитирани како во претходните видеа (тука се бели и во некои моменти се преклопуваат со позадината, а во другите видеа се ем поголеми ем жолти или бели на црна позадина - многу полесно се разликуваат), затоа и се полоши резултатите

примери со и без sharpening атрибутот

In [50]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/ni9b4le5UjA",
    top=0.6,
    bottom=0.75,
    left=0.1,
    right=0.9,
    sharpen=False,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.48s] Ако би можел да смениш нешто
[1.92s] во однос на банките; @дносно амбиентотр
[4.32s] Што прво би сменил?
[5.76s] Дефинитивно ентериерот
[6.72s] да биде нешто ПОкОмОтнОр
[7.2s] да биде нешто покомотно ,
[8.64s] да биде нешто покомотно; НIСШИИ IHI
[9.12s] да биде нешто покомотно,
[9.6s] да укажува на некоја слобода:
[12.0s] Секако; потоа и односот
[13.44s] од банкарите спрема потрошувачите:
[15.84s] TIlI
[16.32s] Би сакала лубето да Fl бидат
[17.76s] повеке трпеливи; да;
[18.72s] повеке да; ке трпеливи;
[19.2s] особено тие што работат таму
[21.12s] Сакам потопли да бидат лућето:
[23.04s] дефинитивно односот: [а;
[24.48s] Односот?
[24.96s] Клиент @O вработен:
[26.4s] Не знам споредмене ОKсе
[26.88s] Незнам; според мене ОK ce
[27.36s] Не знам споредмене ОК се
[27.84s] Кога ке влеземь ретко влагам;
[28.32s] Кога &e влезем; ретко влага[)
[29.28s] Кебиде почеGто ама Сега за сега ОК ce
[29.76s] Кe биде почеGто, ама сега за cerа ОKce
[30.24s] Re биде почесто, ама сега за сега ОК сe
[31.

[{'time': 0.48, 'text': 'Ако би можел да смениш нешто'},
 {'time': 1.92, 'text': 'во однос на банките; @дносно амбиентотр'},
 {'time': 4.32, 'text': 'Што прво би сменил?'},
 {'time': 5.76, 'text': 'Дефинитивно ентериерот'},
 {'time': 6.72, 'text': 'да биде нешто ПОкОмОтнОр'},
 {'time': 7.2, 'text': 'да биде нешто покомотно ,'},
 {'time': 8.64, 'text': 'да биде нешто покомотно; НIСШИИ IHI'},
 {'time': 9.12, 'text': 'да биде нешто покомотно,'},
 {'time': 9.6, 'text': 'да укажува на некоја слобода:'},
 {'time': 12.0, 'text': 'Секако; потоа и односот'},
 {'time': 13.44, 'text': 'од банкарите спрема потрошувачите:'},
 {'time': 15.84, 'text': 'TIlI'},
 {'time': 16.32, 'text': 'Би сакала лубето да Fl бидат'},
 {'time': 17.76, 'text': 'повеке трпеливи; да;'},
 {'time': 18.72, 'text': 'повеке да; ке трпеливи;'},
 {'time': 19.2, 'text': 'особено тие што работат таму'},
 {'time': 21.12, 'text': 'Сакам потопли да бидат лућето:'},
 {'time': 23.04, 'text': 'дефинитивно односот: [а;'},
 {'time': 24.4

In [57]:
extract_captions(
    youtube_url="https://www.youtube.com/shorts/ni9b4le5UjA",
    top=0.6,
    bottom=0.75,
    left=0.1,
    right=0.9,
    sharpen=True,
    save_debug=True
)

Video downloaded!


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[0.48s] Акобиможелда сменишнешто
[1.92s] воодноонабанките @дносно амбиентоц
[3.36s] (о однос[набанките односноамбиентот:
[4.32s] шопрвобисменил?
[5.28s] Шuo прво би сменил?
[5.76s] Дефинитивно ентериерот
[6.72s] да биде нешто покомотно
[9.6s] да укажува на некоја слобода
[12.0s] Секако, потоа односот
[13.44s] Од банкарите спрема потрошувачите.
[15.84s] Tpl
[16.32s] Би сакалалугето да бидат
[17.76s] повеке трпеливи, да,
[18.24s] повек@трпеливи; да;
[18.72s] повеке да, {@Крпеливи;
[19.2s] особенојтие шТО работат таму.
[21.12s] Сакам потопли да' бидат лугето:
[23.04s] Ша;дефинитивно односот_
[23.52s] (а,ефинитивно @ДНОСОТ=
[24.0s] (адефинитивно односот.
[24.48s] Односот?
[24.96s] Клиент cO Вработен:
[25.44s] Клиентсо вработен:
[26.4s] НознамАспоред мене ОК
[26.88s] Нознам; според мено 'ОКсе
[27.36s] Незнам според[ене ;ОК сeb
[27.84s] Kofa ке влезем ретко влагам
[29.28s] (ебиде почесто,ама сега за сега ОК ce
[29.76s] Re биде почесто ама сега за сеra @к
[30.24s] @e биде @ма сега за сe
[30.7

[{'time': 0.48, 'text': 'Акобиможелда сменишнешто'},
 {'time': 1.92, 'text': 'воодноонабанките @дносно амбиентоц'},
 {'time': 3.36, 'text': '(о однос[набанките односноамбиентот:'},
 {'time': 4.32, 'text': 'шопрвобисменил?'},
 {'time': 5.28, 'text': 'Шuo прво би сменил?'},
 {'time': 5.76, 'text': 'Дефинитивно ентериерот'},
 {'time': 6.72, 'text': 'да биде нешто покомотно'},
 {'time': 9.6, 'text': 'да укажува на некоја слобода'},
 {'time': 12.0, 'text': 'Секако, потоа односот'},
 {'time': 13.44, 'text': 'Од банкарите спрема потрошувачите.'},
 {'time': 15.84, 'text': 'Tpl'},
 {'time': 16.32, 'text': 'Би сакалалугето да бидат'},
 {'time': 17.76, 'text': 'повеке трпеливи, да,'},
 {'time': 18.24, 'text': 'повек@трпеливи; да;'},
 {'time': 18.72, 'text': 'повеке да, {@Крпеливи;'},
 {'time': 19.2, 'text': 'особенојтие шТО работат таму.'},
 {'time': 21.12, 'text': "Сакам потопли да' бидат лугето:"},
 {'time': 23.04, 'text': 'Ша;дефинитивно односот_'},
 {'time': 23.52, 'text': '(а,ефинитивно @ДНО